# Treinamento do U-Net (estágio 09)

Executa o protocolo de treino do U-Net (CNN) sobre a divisão espacial k-fold do manifesto (estágio 07), consumindo os patches normalizados pelas estatísticas do estágio 08. Cada dobra treina o modelo com a perda multivariada (Dice + Focal + Boundary), otimizador Adam e scheduler cosine annealing, persistindo pesos, métricas e histórico em `MyDrive/tcc/` de forma idempotente — reexecuções reutilizam os artefatos já existentes.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt`, garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo as pastas de modelos, métricas e histórico de execução.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src import io
from src.data.patch_generation import manifest_path
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Manifesto: {manifest_path(storage_paths)}")
print(f"Modelos: {storage_paths['models']}")
print(f"Métricas: {storage_paths['artifacts_metrics']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch (cudnn determinístico, benchmark desligado e algoritmos determinísticos), além de registrar as versões dos pacotes principais do ambiente — pré-requisitos para o treino bit-reproduzível nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependência dos estágios 06, 07 e 08

Verifica que o manifesto de patches (estágio 06) está disponível e vigente, que a divisão espacial k-fold (estágio 07) preenche a coluna `fold` e que as estatísticas de normalização (estágio 08) estão atualizadas — entradas obrigatórias do treinamento.

In [ ]:
# Verifica as dependências dos estágios 06 (manifesto), 07 (fold) e 08 (estatísticas).
from src.data.eda import normalization_stats_is_current, normalization_stats_path
from src.data.patch_generation import (
    load_manifest,
    manifest_has_folds,
    manifest_is_current,
    manifest_path,
)
from src.data.spatial_split import fold_count
from src.utils import check_dependencies

manifest_file = manifest_path(storage_paths)
check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_file,
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths
        ),
    }
)
manifest = load_manifest(storage_paths)
if not manifest_has_folds(manifest):
    raise ValueError("Coluna fold ausente/incompleta; execute o estágio 07 antes.")
if not normalization_stats_is_current(storage_paths):
    raise ValueError(
        "Estatísticas de normalização desatualizadas; execute o estágio 08 antes."
    )
print(f"Manifesto vigente frente às entradas: {manifest_is_current(storage_paths)}")
print(f"Patches: {len(manifest)} | Dobras: {fold_count()}")

## Modelo U-Net

Constrói o U-Net a partir de `model.unet_channels` do config.yaml, com entrada nas quatro bandas do Sentinel-2 (B2, B3, B4, B8) e saída de um canal de logit binário, exibindo o número de parâmetros treináveis.

In [ ]:
# Constrói o U-Net a partir de config.yaml e exibe o número de parâmetros.
from src.models.unet import build_unet

model = build_unet()
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f"U-Net criado: {n_params:,} parâmetros")
print(f"Canais por nível: {config['model']['unet_channels']}")

## Preparação local dos patches

Garante, no Kaggle, que todos os patches do manifesto tenham cópia no cache local antes do treino, evitando requisições à Drive API durante o carregamento por lote. No Colab/local a operação é um no-op, pois os caminhos já são locais.

In [ ]:
# Garante cópias locais dos patches (Kaggle); no-op no Colab/local.
from src.data.dataset import ensure_patches_local

local_files = ensure_patches_local(storage_paths, manifest)
print(f"Arquivos preparados localmente: {local_files}")

## Treinamento por dobra

Executa o protocolo de treino único (`src/trainer.train_fold`) para cada dobra da validação espacial, construindo um U-Net novo por dobra com a mesma semente. Dobras já treinadas são reutilizadas (idempotência). O protocolo usa a perda multivariada Dice + Focal + Boundary, Adam e cosine annealing — idêntico ao que será aplicado ao SegFormer no estágio 10.

In [ ]:
# Executa o protocolo de treino único por dobra (reutiliza artefatos existentes).
import torch

from src.data.spatial_split import fold_count
from src.models.unet import build_unet
from src.trainer import train_fold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
results = []
for fold in range(fold_count()):
    unet = build_unet()
    result = train_fold(unet, "unet", storage_paths, fold, device)
    results.append(result)

## Resumo do treinamento e artefatos

Exibe o resumo por dobra (melhor época e IoU de validação), persiste o metadata de execução (configuração, fingerprint do manifesto e versões do ambiente) em `artifacts/runs/unet/` e aponta os pesos persistidos.

In [ ]:
# Exibe o resumo do treinamento e persiste o metadata de execução.
from src.trainer import save_run_metadata
from src.utils import print_summary

meta_file = save_run_metadata(storage_paths, "unet")
summary = {"Modelo": "unet", "Dobras treinadas": len(results)}
for result in results:
    tag = " (reutilizada)" if result.skipped else ""
    summary[f"Dobra {result.fold} — IoU val"] = (
        f"{result.val_metrics['iou']:.4f} (época {result.best_epoch}){tag}"
    )
summary["Pesos por dobra"] = str(results[0].weights_path.parent)
summary["Metadata de execução"] = str(meta_file)
print_summary(summary, "09")